[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astronerd21/Deep-Learning-Oil-Slick-Detection/blob/main/Lookalike_Analysis.ipynb)

# False Positive / Lookalike Analysis
**Goal:** Inspect "NO OIL" (0) samples that the model incorrectly predicted as "OIL" (1). This notebook performs an automated evaluation to identify false positives and visualizes their radar signatures.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

%pip install rasterio matplotlib numpy

os.makedirs('/content/data', exist_ok=True)

if not os.path.exists('/content/data/images_s1'):
    print("Extracting satellite images from Drive...")
    !tar -xf /content/drive/MyDrive/OilSlickProject/data/OilSlick/OilSlick-images_s1-00.tar -C /content/data/
    !tar -xf /content/drive/MyDrive/OilSlickProject/data/OilSlick/OilSlick-images_s1-01.tar -C /content/data/
    print("Extraction complete!")
else:
    print("Images are already extracted.")

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Paths to the finished evaluations
fp_paths = {
    "Baseline Random": Path("/content/drive/MyDrive/OilSlickProject/results/false_positives_baseline_random.txt"),
    "ResNet Geo OOD": Path("/content/drive/MyDrive/OilSlickProject/results/false_positives_geographic_ood.txt"),
    "TerraMind Geo OOD": Path("/content/drive/MyDrive/OilSlickProject/results/false_positives_terramind_ood.txt")
}

# Function to read the lists
def load_fp_list(path):
    if not path.exists():
        print(f"WARNING: File not found -> {path}")
        return []
    with open(path, "r") as f:
        return [line.strip() for line in f if line.strip()]

# Load lists into a dictionary
fps = {name: load_fp_list(path) for name, path in fp_paths.items()}

for name, ids in fps.items():
    print(f"{name}: {len(ids)} False Positives loaded.")

# Helper functions for plotting
data_dir = Path("/content/data/images_s1/")

def load_and_normalize_sar(filepath: Path) -> np.ndarray:
    with rasterio.open(filepath) as src:
        data = src.read().astype(np.float32)
    for c in range(data.shape[0]):
        channel_mean = np.mean(data[c])
        channel_std = np.std(data[c])
        if channel_std > 0:
            data[c] = (data[c] - channel_mean) / channel_std
    return data

def plot_top_10(model_name):
    ids = fps.get(model_name, [])[:10]
    if not ids:
        return
    
    num_samples = len(ids)
    fig, axes = plt.subplots(num_samples, 4, figsize=(20, 4 * num_samples))
    fig.suptitle(f"Lookalike Analysis: {model_name} (Top {num_samples})", fontsize=16, y=1.02)

    for idx, filename in enumerate(ids):
        # Fallback in case the .txt files are missing the .tif extension
        img_file = filename if filename.endswith('.tif') else f"{filename}_s1.tif"
        filepath = data_dir / img_file
        
        try:
            data = load_and_normalize_sar(filepath)
            axes[idx, 0].imshow(data[0], cmap='gray'); axes[idx, 0].set_title(f"VV | {filename}"); axes[idx, 0].axis('off')
            axes[idx, 1].hist(data[0].ravel(), bins=50, color='blue', alpha=0.7); axes[idx, 1].set_title("VV Hist")
            axes[idx, 2].imshow(data[1], cmap='gray'); axes[idx, 2].set_title(f"VH | {filename}"); axes[idx, 2].axis('off')
            axes[idx, 3].hist(data[1].ravel(), bins=50, color='orange', alpha=0.7); axes[idx, 3].set_title("VH Hist")
        except Exception as e:
            axes[idx, 0].set_title(f"Error at {filename}"); axes[idx, 0].axis('off')
            print(f"Error loading {filename}: {e}")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_top_10("Baseline Random")

In [ ]:
plot_top_10("ResNet Geo OOD")

In [ ]:
plot_top_10("TerraMind Geo OOD")

In [ ]:
# Convert lists to sets to find the intersection
sets = {name: set(ids) for name, ids in fps.items()}

# Intersection: Images that were a False Positive in ALL THREE models
uncommon_lookalikes = sets["Baseline Random"] & sets["ResNet Geo OOD"] & sets["TerraMind Geo OOD"]

print("=== OVERLAP ANALYSIS ===")
print(f"Number of 'impossible lookalikes' (False alarm in all 3 models): {len(uncommon_lookalikes)}\n")

print("Top 10 IDs of these extremely hard cases:")
for img_id in list(uncommon_lookalikes)[:10]:
    print(f" - {img_id}")

# visualize the hardest case 
if uncommon_lookalikes:
    hard_case = list(uncommon_lookalikes)[0]
    print(f"\nVisualizing an exemplary hard case: {hard_case}")
    fps["Hardcase Demo"] = [hard_case]
    plot_top_10("Hardcase Demo")